In [1]:
import gc
# import torch

gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [43]:
import json
from pathlib import Path
import pandas as pd
import numpy as np


baselines = ["md3po", "md3po_sac", "ddpo", "b2diffurl", "dpok"]
eval_path = Path("outputs/")
training_data_eval_file = "training_metrics.json"
evaluation_data_eval_file = "eval_metrics.json"

def get_training_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    all_metric_entries = []
    for entry in list_of_metrics:
        all_metric_entries.append(entry['metrics'])
    df = pd.DataFrame(all_metric_entries)
    return df

def get_eval_metrics_df(file_path):
    if file_path.exists():
        with open(file_path, "r") as f:
            list_of_metrics = json.load(f)
    else:
        print(f"File {file_path} does not exist. Skipping.")
        return pd.DataFrame()  # Return an empty DataFrame if the file doesn't exist
    
    scores = ['bert_reward', 'clip_reward']
    eval_summary = list()
    for entry in list_of_metrics:
        for score in scores:
            eval_summary.append({
                "value": round(np.array(entry[score]).mean(), 4),
                "score": score,
                "epoch": entry['epoch']
            })
    eval_df = pd.DataFrame(eval_summary)
    return eval_df
    
def get_metrics():
    training_eval_df = pd.DataFrame()
    eval_df = pd.DataFrame()
    for baseline in baselines:
        baseline_path = eval_path / baseline
        training_data_eval_file_path = baseline_path / "training_evals" / training_data_eval_file if baseline != "md3po_sac" else baseline_path / "seed_123/training_evals" / training_data_eval_file
        eval_data_file_path = baseline_path / "evals" / evaluation_data_eval_file if baseline != "md3po_sac" else baseline_path / "seed_123/evals" / evaluation_data_eval_file

        
        df = get_training_metrics_df(training_data_eval_file_path)
        df["method"] = baseline
        training_eval_df = pd.concat([training_eval_df, df], ignore_index=True)

        df = get_eval_metrics_df(eval_data_file_path)
        df["method"] = baseline
        eval_df = pd.concat([eval_df, df], ignore_index=True)
    return training_eval_df, eval_df

In [ ]:
training_eval_df, eval_df = get_metrics()

import matplotlib.pyplot as plt

temp_eval_df = eval_df[eval_df['score'] == 'clip_reward']
temp_eval_df = temp_eval_df[temp_eval_df['method'].isin(list(set(baselines)-set(["dpok_old"])))]
eval_summary = temp_eval_df.pivot_table(
    index=['method', 'score'],
    columns='epoch',
    values='value',
)
# eval_summary.T.iloc[:, 0].plot()
# plt.title("Bert Reward")
# plt.show()
# eval_summary.T.iloc[:, 1].plot()
# plt.title("Clip Reward")
# plt.show()

transposed_eval_summary = eval_summary.T
transposed_eval_summary = transposed_eval_summary.reset_index().droplevel(level=1, axis=1)
# transposed_eval_summary[transposed_eval_summary.index.isin([2]+list(range(0, 50, 5)))].rolling(window=2).mean().round(3).plot(marker='o')
transposed_eval_summary['queries'] = transposed_eval_summary['epoch'].astype(int)*256
transposed_eval_summary.pivot_table(
    index='queries',
    values=['b2diffurl', 'ddpo', 'md3po', 'md3po_sac'],
)#.rolling(window=1).mean().plot(marker='*')

In [45]:
cols = ['advantage_mean', 'approx_kl', 'current_reward_mean', 'current_reward_std', 'epoch', 'loss', 'parameter_update_norm_mean',
    'raw_reward_mean', 'raw_reward_std', 'replay_reward_mean',
    'replay_reward_std', 'reward_mean', 'reward_std', 'selected_samples', 'skipped_updates','method']
training_eval_df[cols].pivot_table(
    index=['epoch'],
    columns=['method'],
    values=['reward_mean'], #'loss', 
    # aggfunc='mean'
).round(3)#.rolling(window=1).mean().plot(kind='line') ##

reward_mean                               
method   b2diffurl   ddpo   dpok  md3po md3po_sac
epoch                                            
1            0.324  0.324  0.325  0.331      0.32
2            0.324  0.325  0.324  0.327       NaN
3            0.326  0.325    NaN  0.333       NaN
4            0.328  0.332    NaN  0.337       NaN
5            0.328  0.335    NaN  0.340       NaN
6            0.328  0.334    NaN  0.339       NaN
7            0.332  0.334    NaN  0.343       NaN
8            0.328  0.335    NaN  0.344       NaN
9            0.328  0.330    NaN  0.346       NaN
10           0.336  0.339    NaN  0.350       NaN
11           0.329  0.335    NaN  0.345       NaN
12           0.330  0.338    NaN  0.345       NaN
13           0.324  0.331    NaN  0.345       NaN
14           0.333  0.339    NaN  0.342       NaN
15           0.328  0.335    NaN  0.347       NaN
16           0.331  0.340    NaN  0.351       NaN
17           0.325  0.331    NaN  0.343       NaN
18           0.331  0.338    NaN  0.348       NaN
19           0.329  0.337    NaN  0.344       NaN
20           0.330  0.342    NaN  0.342       NaN
21           0.332  0.341    NaN  0.341       NaN
22           0.329  0.341    NaN  0.339       NaN
23           0.329  0.344    NaN  0.345       NaN
24           0.334  0.344    NaN  0.344       NaN
25           0.332  0.344    NaN  0.337       NaN
26           0.328  0.342    NaN  0.337       NaN
27           0.335  0.346    NaN  0.342       NaN
28           0.337  0.350    NaN  0.341       NaN
29           0.338  0.347    NaN  0.341       NaN
30           0.330  0.341    NaN  0.339       NaN
31           0.336  0.348    NaN  0.342       NaN
32           0.334  0.345    NaN  0.337       NaN
33           0.336  0.351    NaN  0.344       NaN
34           0.333  0.344    NaN  0.340       NaN
35           0.338  0.349    NaN  0.339       NaN
36           0.337  0.350    NaN  0.338       NaN
37           0.339  0.354    NaN  0.343       NaN
38           0.342  0.353    NaN  0.339       NaN
39           0.344  0.353    NaN  0.341       NaN
40           0.341  0.350    NaN  0.344       NaN
41           0.344  0.349    NaN  0.341       NaN
42           0.347  0.357    NaN  0.337       NaN
43           0.354  0.356    NaN  0.344       NaN
44           0.353  0.355    NaN  0.342       NaN
45           0.348  0.357    NaN  0.339       NaN
46           0.344  0.353    NaN  0.337       NaN
47           0.346  0.350    NaN  0.337       NaN
48           0.348  0.355    NaN  0.341       NaN
49           0.351  0.360    NaN  0.338       NaN
50           0.355  0.355    NaN  0.345       NaN